# 📚 PDF Kitap → Türkçe Çeviri (Ücretsiz GPU)

Bu defter çeviriyi **Google'ın ücretsiz GPU'sunda** yapar; senin cihazın (tablet/telefon) sadece başlatır.
600 sayfalık bir kitap genelde **birkaç dakika** sürer.

## Kullanım (sırayla)
1. Üstte **Çalışma zamanı → Çalışma zamanı türünü değiştir → Donanım hızlandırıcı: GPU** seç, kaydet.
2. Her hücrenin solundaki **▶ (oynat)** düğmesine **yukarıdan aşağıya sırayla** bas.
3. 2. hücrede PDF'ini yükle. Son hücrede çevrilmiş PDF iner.

> Not: Model seçimi 4. hücrede. Varsayılan en iyi kalite içindir.

## 1) Kurulum (gerekli kütüphaneler + GPU kontrolü)

In [ ]:
!pip -q install transformers sentencepiece sacremoses pymupdf fpdf2 langdetect tqdm
import torch
if torch.cuda.is_available():
    print('✓ GPU hazır:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU YOK! Üstten: Çalışma zamanı → Türünü değiştir → GPU seç, sonra bu hücreyi tekrar çalıştır.')

## 2) PDF'ini yükle

In [ ]:
from google.colab import files
up = files.upload()
PDF_PATH = list(up.keys())[0]
print('Yüklendi:', PDF_PATH)

## 3) Metni çıkar (sayfa sayfa, paragraf blokları)

In [ ]:
import fitz  # pymupdf
import re

doc = fitz.open(PDF_PATH)
pages = []  # her sayfa: paragraf metinleri listesi
for page in doc:
    blocks = page.get_text('blocks')  # (x0,y0,x1,y1, text, no, type)
    texts = []
    for b in sorted(blocks, key=lambda b: (round(b[1]), b[0])):
        if b[6] != 0:  # 0 = metin bloğu
            continue
        t = b[4]
        t = re.sub(r'(\w)-\n(\w)', r'\1\2', t)   # satır sonu tire birleştir
        t = re.sub(r'\s+', ' ', t).strip()
        if t:
            texts.append(t)
    pages.append(texts)

n_blocks = sum(len(p) for p in pages)
print(f'Sayfa: {len(pages)} | toplam paragraf: {n_blocks}')
print('Örnek:', (pages[0][0][:120] + '...') if pages and pages[0] else '(boş)')

## 4) Modeli yükle (GPU'da)

- **Varsayılan (en iyi kalite):** `facebook/nllb-200-distilled-1.3B` — kişisel kullanım için ideal.
- **Ticari dağıtım yapacaksan** (lisans derdi olmasın): aşağıdaki satırı `facebook/m2m100_1.2B` yap (MIT lisansı).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = 'facebook/nllb-200-distilled-1.3B'   # ticari için: 'facebook/m2m100_1.2B'
IS_NLLB = 'nllb' in MODEL

print('Model indiriliyor (ilk seferde birkaç dk):', MODEL)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16).to('cuda').eval()
print('✓ Model hazır.')

## 5) Kaynak dili algıla

In [ ]:
from langdetect import detect

# langdetect (ISO 639-1) -> NLLB FLORES / M2M kodları
NLLB = {'en':'eng_Latn','tr':'tur_Latn','de':'deu_Latn','fr':'fra_Latn','es':'spa_Latn','it':'ita_Latn','pt':'por_Latn','ru':'rus_Cyrl','ar':'arb_Arab','zh':'zho_Hans','ja':'jpn_Jpan','ko':'kor_Hang','nl':'nld_Latn','pl':'pol_Latn','uk':'ukr_Cyrl','ro':'ron_Latn','el':'ell_Grek','fa':'pes_Arab','sv':'swe_Latn','fi':'fin_Latn','cs':'ces_Latn','hu':'hun_Latn'}
TGT = 'tur_Latn' if IS_NLLB else 'tr'

sample = ' '.join(t for p in pages for t in p)[:3000]
try:
    iso = detect(sample)
except Exception:
    iso = 'en'
SRC = NLLB.get(iso, 'eng_Latn') if IS_NLLB else iso
print('Algılanan kaynak dil:', iso, '->', SRC)

## 6) Çevir (GPU'da, toplu + tekrar eden bloklar bir kez)

In [ ]:
import re, torch
from collections import Counter
from tqdm.auto import tqdm

# ===== AYARLAR =====
NUM_BEAMS = 4        # kalite (2 daha hızlı)
BATCH = 16           # GPU belleği taşarsa 8'e düşür
MAX_LEN = 512
EXTRA_PROTECT = []   # elle korumak istediğin isimler: ['Car Askand', 'Mog-Pharau']
DONT_PROTECT  = []   # yanlış korunursa çevrilsin diye buraya: ['Holy War', 'Three Seas']

# ===== 1) Korunacak özel isimleri otomatik bul =====
COMMON = set("The A An And Or But Not No Yes In On At To Of For With From By As Is Are Was Were Be Been Has Have Had Do Does Did He She It They We You I Him Her Them His Hers Their Our Your My Me This That These Those When Where What Who Why How Which While Then Than If So All Any Each Every Some Such Chapter Part Book Volume Prologue Epilogue Appendix One Two Three Four Five Six Seven Eight Nine Ten Mr Mrs Ms Dr".split())
src_all = '\n'.join(t for p in pages for t in p)
cand = re.findall(r"\b[A-Z][A-Za-zâêîôûäöü'’-]+(?:[ ][A-Z][A-Za-zâêîôûäöü'’-]+)*\b", src_all)
freq = Counter(cand)
PROTECT = set(EXTRA_PROTECT)
for name, c in freq.items():
    if name in DONT_PROTECT:
        continue
    parts = name.split()
    if len(parts) == 1 and (name in COMMON or c < 2):
        continue
    PROTECT.add(name)
PROTECT = sorted(PROTECT, key=len, reverse=True)   # uzun isimler önce
print(len(PROTECT), 'isim korunacak. Örnek:', PROTECT[:20])

# ===== 2) Maskele / geri yükle =====
idx_map = {i: n for i, n in enumerate(PROTECT)}
name_to_tok = {n: f'Qx{i}xQ' for i, n in enumerate(PROTECT)}
mask_pat = re.compile('|'.join(re.escape(n) for n in PROTECT)) if PROTECT else None
restore_pat = re.compile(r'Q\s*x\s*(\d+)\s*x\s*Q', re.IGNORECASE)
def mask(s):   return mask_pat.sub(lambda m: name_to_tok[m.group(0)], s) if mask_pat else s
def unmask(s): return restore_pat.sub(lambda m: idx_map.get(int(m.group(1)), m.group(0)), s)

# ===== 3) Çevir =====
tok.src_lang = SRC
forced = tok.convert_tokens_to_ids(TGT) if IS_NLLB else tok.get_lang_id(TGT)
def translate(texts):
    out = []
    for i in tqdm(range(0, len(texts), BATCH)):
        chunk = [mask(t) for t in texts[i:i+BATCH]]
        enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to('cuda')
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=forced, max_length=MAX_LEN, num_beams=NUM_BEAMS)
        out += [unmask(x) for x in tok.batch_decode(gen, skip_special_tokens=True)]
    return out

# Tüm blokları düzleştir, tekrar edenleri tekilleştir (üstbilgi/altbilgi)
flat = [(pi, bi, t) for pi, blocks in enumerate(pages) for bi, t in enumerate(blocks)]
texts = [t for _, _, t in flat]
uniq = list(dict.fromkeys(texts))
print(f'{len(texts)} paragraf, {len(uniq)} benzersiz — çeviriliyor...')
tr_map = dict(zip(uniq, translate(uniq)))

out_pages = [['' for _ in blocks] for blocks in pages]
for (pi, bi, t) in flat:
    out_pages[pi][bi] = tr_map[t]
print('✓ Çeviri tamam.')

## 7) Türkçe PDF oluştur ve indir

In [ ]:
from fpdf import FPDF
import matplotlib.font_manager as fm

FONT = fm.findfont('DejaVu Sans')  # Colab'da her zaman bulunur
OUT = 'ceviri-turkce.pdf'

pdf = FPDF(format='A4')
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_font('DejaVu', '', FONT)
pdf.set_font('DejaVu', size=11)

for blocks in out_pages:
    pdf.add_page()
    for para in blocks:
        if para.strip():
            pdf.multi_cell(0, 7, para)
            pdf.ln(2)

pdf.output(OUT)
print('✓ PDF hazır:', OUT)
from google.colab import files
files.download(OUT)